In [ ]:
import json
import numpy as np
import pandas as pd
import pycountry_convert as pc
from IPython.display import display, Markdown
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, FULL_RUN, OXCGRT_COLMAP
from emu_renewal.outputs import get_policy_weight_summary
from emu_renewal.plotting import plot_policy_weight_heatmap
from emu_renewal.utils import get_analysis_paths, get_analysis_commits_df, \
    get_countries_by_continent, get_country_name, sort_countries_by_name

set_matplotlib_formats("svg")

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json", "r"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
summary = get_policy_weight_summary(analysis_paths)

supported = summary.index[summary["prop_better"] > 0.5]
unsupported = summary.index[summary["prop_better"] <= 0.5]
countries_by_cont = get_countries_by_continent(list(supported))
# Although Singapore is grouped with Oceania epidemiologically,
# results are shown with Asia becuase it is the only country included in this category
countries_by_cont["OC"].remove("SGP")
countries_by_cont["AS"] = sort_countries_by_name(countries_by_cont["AS"] + ["SGP"])
countries_by_cont = {k: v for k, v in countries_by_cont.items() if v}

# Shared across the continent panels, so that colours are comparable between them
max_dev = (summary.loc[supported, OXCGRT_COLMAP["custom"]] - 0.5).abs().max().max()
dev_limit = float(np.ceil(max_dev * 20.0) / 20.0)

# Purpose
This document presents the calibrated weight for each OxCGRT policy domain
of the floored OxCGRT analysis as a country by policy matrix.

Each weight $\omega_{p}$ has a uniform prior over $[0, 1]$,
so the prior median of every weight parameter is 0.5.
We therefore plot the posterior median minus 0.5,
such that the colour scale is centred on the value that would be obtained
for a policy about which the calibration is uninformative.
Red indicates weights revised upwards and blue those revised downwards.
We present the weights before normalisation because normalisation
makes each value depend on the values taken by the other seven policies.

Only countries for which residual-process dispersion improved relative to
`no_scaling` are included, using the same criterion as in the preceding documents.
For the remaining countries, the analysis provides no evidence that
policy contributed to transmission, such that the weights should not be interpreted
and so are not displayed. Singapore is shown with Asia here, although
the epidemiological implementation is aligned with Oceania otherwise. 
This is because it is the only country of this group for which 
the dispersion posterior criterion was reached (Australia and New Zealand excluded).

The final column repeats the composite scaling strength, $1-f^{m}$,
for the same country. This gives a sense for the overall strength of the policy effects,
but is presented in greater detail in Supplement 09. Supplement 10 provides 
the full distribution of the policy weights (which are not always strongly identified).

{{< pagebreak >}}

# Results by continent

In [ ]:
for cont, countries in countries_by_cont.items():
    display(Markdown(f"## {pc.convert_continent_code_to_continent_name(cont)}"))
    display(plot_policy_weight_heatmap(summary, countries, max_dev=dev_limit))
    display(Markdown("{{< pagebreak >}}"))

# Countries omitted from the matrix
The following countries were analysed under the floored OxCGRT approach,
but did not show an improvement in the dispersion of the residual
transmission process relative to the analysis without scaling.
Their policy weights are omitted above, and the proportion of paired runs
with lower dispersion than baseline is given for reference.

In [ ]:
omitted = summary.loc[unsupported, ["strength", "prop_better"]].copy()
omitted.index = omitted.index.map(get_country_name)
omitted.columns = ["effect strength", "proportion better"]
display(Markdown(omitted.sort_index().round(3).to_markdown()))

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())